In [0]:
# ============================================================
# Silver — Source 06: Stripe API Payments
#
# Transformations:
#   - Cast order_id STRING to LONG
#   - Cast customer_id STRING to LONG
#   - Cast created_at ISO string to timestamp
#   - Uppercase currency
#   - Validate amount > 0
#   - Reject null payment_intent_id → quarantine
#   - order_id null is valid (some charges not linked to orders)
#   - Deduplicate on payment_intent_id
#
# Source:  bronze.src_06_payments.charges
# Target:  silver.src_06_payments.charges
# Quarantine: silver.quarantine.src_06_payments
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_06_payments.charges'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_06_payments'

VALID_STATUSES = ['succeeded', 'failed', 'pending', 'refunded', 'disputed', 'canceled']

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_06_payments')
print('Silver Source 06 Stripe — starting...')


In [0]:
# ── LOAD AND CLEAN ────────────────────────────────────────────
bronze = spark.table(f'{BRONZE_CATALOG}.src_06_payments.charges')
total = bronze.count()
print(f'Bronze rows: {total}')

# Step 1: Cast string IDs to long, timestamps, normalise
df = bronze \
    .withColumn('order_id',    F.col('order_id').cast('long')) \
    .withColumn('customer_id', F.col('customer_id').cast('long')) \
    .withColumn('created_at',  F.to_timestamp(F.col('created_at'))) \
    .withColumn('currency',    F.upper(F.trim(F.col('currency')))) \
    .withColumn('status',      F.lower(F.trim(F.col('status'))))

# Step 2: Bad rows
# order_id null is VALID — some Stripe charges not linked to orders
bad = df.filter(
    F.col('payment_intent_id').isNull() |
    F.col('amount').isNull() |
    (F.col('amount') <= 0) |
    F.col('created_at').isNull() |
    ~F.col('status').isin(VALID_STATUSES)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('charges'))

# Step 3: Good rows
good = df.filter(
    F.col('payment_intent_id').isNotNull() &
    F.col('amount').isNotNull() &
    (F.col('amount') > 0) &
    F.col('created_at').isNotNull() &
    F.col('status').isin(VALID_STATUSES)
)

w = Window.partitionBy('payment_intent_id').orderBy(F.col('created_at').desc())
good = good.withColumn('_rn', F.row_number().over(w)) \
           .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad.count()
good_count = good.count()
print(f'Stripe charges: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Show status distribution
df.groupBy('status').count().orderBy('count', ascending=False).show()

# Step 4: Write
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.payment_intent_id = s.payment_intent_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('MERGE complete')
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
    print('Initial load complete')

# Step 5: Quarantine
if bad_count > 0:
    quarantine = bad.select(
        F.lit('src_06_payments').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    )
    quarantine.write.format('delta').mode('append') \
        .option('mergeSchema', 'true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} rows quarantined')
else:
    print('No quarantine rows')


In [0]:
# ── VERIFY ────────────────────────────────────────────────────
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_06_payments.charges: {count} rows')
spark.sql(f"""
    SELECT status, COUNT(*) as cnt, SUM(amount) as total_amount
    FROM {TARGET_TABLE}
    GROUP BY status
    ORDER BY cnt DESC
""").show()
